# Break-My-VLA — M2 Real LIBERO + SmolVLA Chaos Benchmark

1. 在右上角 **Runtime → Change runtime type** 选择 **G4 / L4 / A100** GPU。
2. 点击 **Runtime → Run all**，会自动挂载 Google Drive、解压 `Break-My-VLA-m2.zip`、用 `uv` 建立干净的 Python 3.11 + CUDA 12.8 环境并运行 4 组真实 LIBERO + SmolVLA 对照实验（`baseline` / `object_nudge` / `lighting_shift` / `int8_low_steps`）。
3. 结果 GIF 和 `metrics.json` 会自动保存到你的 Google Drive：`MyDrive/Break-My-VLA/results/m2_smoke/`。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile


def run_cmd(cmd, **kwargs):
    print('>>>', ' '.join(str(x) for x in cmd), flush=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        **kwargs,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

DRIVE_DIR = Path('/content/drive/MyDrive/Break-My-VLA')
REPO_ZIP = DRIVE_DIR / 'Break-My-VLA-m2.zip'
WORKSPACE = Path('/content/break-my-vla-m2')
VENV_DIR = WORKSPACE / 'venv-py311'
OUT_DIR = DRIVE_DIR / 'results' / 'm2_smoke'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert REPO_ZIP.is_file(), f'找不到 {REPO_ZIP}，请确认 Drive 路径'

# 1. 解压最新代码
WORKSPACE.mkdir(parents=True, exist_ok=True)
with ZipFile(REPO_ZIP) as zf:
    zf.extractall(WORKSPACE)

# 2. 用 uv 建独立 Python 3.11 环境（绕过 Colab 缺 python3.13-venv，全量预编译 wheel）
if shutil.which('uv') is None:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

python = VENV_DIR / 'bin' / 'python'
if not python.exists():
    run_cmd(['uv', 'venv', '--python', '3.11', str(VENV_DIR)])

print('Installing CUDA 12.8 PyTorch + LeRobot + LIBERO...', flush=True)
run_cmd([
    'uv', 'pip', 'install', '--python', str(python),
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    'torch==2.10.0', 'torchvision==0.25.0',
    'lerobot[smolvla,libero]==0.4.4',
    'hf-libero==0.1.5',
    'mujoco==3.3.7',
    'transformers==4.57.6',
    'fastapi>=0.115,<1', 'uvicorn>=0.35,<1', 'httpx>=0.27,<1',
    'pydantic>=2.8,<3', 'pillow>=10.3,<13',
])

sys.path.insert(0, str(WORKSPACE))
from scripts.colab_m2_smolvla import rendering_backend, source_archive  # noqa: E402

renderer = rendering_backend(python)
src_sha = hashlib.sha256(source_archive()).hexdigest()
print(f'Renderer={renderer}, Source SHA256={src_sha[:12]}', flush=True)

# 3. 运行 4 组真实 LIBERO + SmolVLA 对照实验并直接写入 Drive
cmd = [str(python), '-m', 'benchmarks.m2_smoke', '--output', str(OUT_DIR)]
env = os.environ | {
    'MUJOCO_GL': renderer,
    'LIBERO_CONFIG_PATH': str(WORKSPACE / 'libero-config'),
    'BMVLA_SOURCE_SHA256': src_sha,
    'PYTHONUNBUFFERED': '1',
    'TOKENIZERS_PARALLELISM': 'false',
}
run_cmd(cmd, cwd=WORKSPACE, env=env)
print('完成！输出目录：', OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(f'  - {p.name} ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

OUT_DIR = Path('/content/drive/MyDrive/Break-My-VLA/results/m2_smoke')
metrics = json.loads((OUT_DIR / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2, ensure_ascii=False))

for gif_name in ['baseline.gif', 'object_nudge.gif', 'lighting_shift.gif', 'int8_low_steps.gif']:
    p = OUT_DIR / gif_name
    if p.exists():
        display(Markdown(f'### {gif_name}'))
        display(Image(filename=str(p)))